# Lab 03 — Embeddings & Vector Distance Functions

Embeddings convert text into numerical vectors for similarity search. Vector distances measure how close two vectors are.

| Item | Detail |
|---|---|
| **Functions** | `EMBED_TEXT_768`, `EMBED_TEXT_1024`, `VECTOR_COSINE_SIMILARITY`, `VECTOR_INNER_PRODUCT`, `VECTOR_L1_DISTANCE`, `VECTOR_L2_DISTANCE` |
| **Exam Domain** | 2.0 Gen AI & LLM Functions |
| **You'll learn** | Embedding models, vector dimensions, all 4 distance/similarity metrics |


## Step 1 — Environment Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — EMBED_TEXT_768 vs EMBED_TEXT_1024

| Function | Dimensions | Trade-off |
|---|---|---|
| `EMBED_TEXT_768` | 768 | Lighter, faster, lower storage cost |
| `EMBED_TEXT_1024` | 1024 | Higher fidelity, better for complex semantic tasks |

Both use the `e5-base-v2` model and return a VECTOR type.


In [ ]:
SELECT
    'EMBED_TEXT_768' AS function_name,
    VECTOR_DIMENSION(SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', 'Snowflake data warehouse')) AS dimensions
UNION ALL
SELECT
    'EMBED_TEXT_1024',
    VECTOR_DIMENSION(SNOWFLAKE.CORTEX.EMBED_TEXT_1024('e5-base-v2', 'Snowflake data warehouse'));

## Step 3 — All Vector Distance Functions

| Function | What It Measures | Range | Best For |
|---|---|---|---|
| `VECTOR_COSINE_SIMILARITY` | Angular similarity | -1 to 1 (1 = identical) | Text similarity, RAG |
| `VECTOR_INNER_PRODUCT` | Dot product | Unbounded (higher = more similar) | Ranking, recommendation |
| `VECTOR_L1_DISTANCE` | Manhattan distance | 0+ (lower = more similar) | Sparse features |
| `VECTOR_L2_DISTANCE` | Euclidean distance | 0+ (lower = more similar) | Geometric similarity |


In [ ]:
WITH embeddings AS (
    SELECT
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', 'cloud data warehouse') AS vec_a,
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', 'data storage platform') AS vec_b,
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', 'chocolate cake recipe') AS vec_c
)
SELECT
    'warehouse vs platform (similar)' AS comparison,
    ROUND(VECTOR_COSINE_SIMILARITY(vec_a, vec_b), 4) AS cosine_sim,
    ROUND(VECTOR_INNER_PRODUCT(vec_a, vec_b), 4) AS inner_product,
    ROUND(VECTOR_L1_DISTANCE(vec_a, vec_b), 4) AS l1_distance,
    ROUND(VECTOR_L2_DISTANCE(vec_a, vec_b), 4) AS l2_distance
FROM embeddings
UNION ALL
SELECT
    'warehouse vs cake (dissimilar)',
    ROUND(VECTOR_COSINE_SIMILARITY(vec_a, vec_c), 4),
    ROUND(VECTOR_INNER_PRODUCT(vec_a, vec_c), 4),
    ROUND(VECTOR_L1_DISTANCE(vec_a, vec_c), 4),
    ROUND(VECTOR_L2_DISTANCE(vec_a, vec_c), 4)
FROM embeddings;

## Step 4 — Storing Embeddings in a Table

In practice, you compute embeddings once and store them for repeated search.

In [ ]:
CREATE OR REPLACE TABLE embedding_store (
    text_id INT AUTOINCREMENT,
    source_text TEXT,
    embedding VECTOR(FLOAT, 768)
);

INSERT INTO embedding_store (source_text, embedding)
SELECT column1, SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', column1)
FROM VALUES
    ('Snowflake is a cloud data warehouse'),
    ('Machine learning models require training data'),
    ('SQL is used to query databases'),
    ('Cortex AI provides LLM functions in SQL'),
    ('Chocolate chip cookies are delicious');

SELECT text_id, source_text FROM embedding_store;

## Step 5 — Nearest-Neighbor Search

In [ ]:
SELECT
    source_text,
    ROUND(VECTOR_COSINE_SIMILARITY(
        embedding,
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', 'How do I run queries on Snowflake?')
    ), 4) AS similarity
FROM embedding_store
ORDER BY similarity DESC;

## Key Takeaways

- `EMBED_TEXT_768` is lighter/faster; `EMBED_TEXT_1024` is higher fidelity.
- **Cosine similarity** is the go-to metric for text similarity and RAG.
- **Inner product** works well for ranking when vectors are normalized.
- **L1/L2 distances** measure geometric distance (lower = more similar).
- Store embeddings in VECTOR columns for efficient repeated search.
